# Часть C. Модели и данные

Для этой части GPU не нужна: Runtime → Change runtime type → CPU (так не тратятся compute units).
Подробный разбор каждой строки — в файле [docs/C_explained.md](https://github.com/IvanovskyDev/Machine-Unlearning-in-LLM/blob/main/docs/C_explained.md).

**1. Старт сессии** — то же, что шаг 1 части B: Drive, папки, переменные окружения, токен Hugging Face.

In [ ]:
import os                                   # папки и переменные окружения

from google.colab import drive, userdata    # Google Drive и секреты Colab

drive.mount("/content/drive")               # подключить Google Drive

DRIVE = "/content/drive/MyDrive/unlearning_data"   # папка проекта на Drive (постоянная)
FAST = "/content/fast"                             # папка на диске машины (очищается после сессии)

os.makedirs(DRIVE + "/saves", exist_ok=True)       # чекпоинты моделей
os.makedirs(DRIVE + "/results_raw", exist_ok=True) # сырые результаты атак
os.makedirs(DRIVE + "/envs", exist_ok=True)        # lock-файлы окружений и моделей
os.makedirs(FAST + "/hf_home", exist_ok=True)      # кэш Hugging Face
os.makedirs(FAST + "/models", exist_ok=True)       # скачанные модели

os.environ["BIG"] = DRIVE                          # «большой диск» из плана
os.environ["HF_HOME"] = FAST + "/hf_home"          # кэш Hugging Face
os.environ["MODELS"] = FAST + "/models"            # папка моделей
os.environ["TOKENIZERS_PARALLELISM"] = "false"     # меньше лишних предупреждений
os.environ["PYTHONUNBUFFERED"] = "1"               # вывод программ сразу попадает в лог
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # токен Hugging Face из секрета HF_TOKEN

print("Старт сессии выполнен")

**2. Смотрим модель на Hugging Face** (блок 11): её ревизию и список файлов с размерами. Сама страница модели: https://huggingface.co/open-unlearning/tofu_Llama-3.2-1B-Instruct_full

In [ ]:
from huggingface_hub import HfApi   # доступ к Hugging Face из Python

api = HfApi()
info = api.model_info("open-unlearning/tofu_Llama-3.2-1B-Instruct_full", files_metadata=True)

print("Ревизия:", info.sha)          # хэш версии модели: по нему модель закрепляется
for file in info.siblings:          # siblings — список файлов модели
    print(file.rfilename, "—", round(file.size / 1e6, 1), "МБ")

**3. Закрепляем ревизии моделей плана** в файле `envs/models.lock.json` на Drive (блок 12). Уже закреплённые ревизии не меняются, поэтому при каждой загрузке приходит та же версия модели.

In [ ]:
import json                         # чтение и запись файлов JSON

from huggingface_hub import HfApi

api = HfApi()
LOCK = DRIVE + "/envs/models.lock.json"

# модели плана до ноября (блок 10); новые модели дописываются в этот список
REPOS = [
    "open-unlearning/tofu_Llama-3.2-1B-Instruct_full",
    "open-unlearning/tofu_Llama-3.2-1B-Instruct_retain99",
    "open-unlearning/tofu_Llama-3.2-3B-Instruct_full",
    "open-unlearning/tofu_Llama-3.2-3B-Instruct_retain95",
    "open-unlearning/tofu_Llama-3.2-3B-Instruct_retain99",
    "open-unlearning/tofu_Llama-3.2-3B-Instruct_retain90",
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen2.5-14B-Instruct-AWQ",
    "sentence-transformers/all-mpnet-base-v2",
]

# прочитать уже закреплённые ревизии, если файл есть
lock = {}
if os.path.exists(LOCK):
    with open(LOCK) as f:
        lock = json.load(f)

# узнать ревизию только у тех моделей, которых ещё нет в файле
for repo in REPOS:
    if repo in lock:
        print("уже закреплена:", repo, lock[repo][:7])   # [:7] — первые 7 знаков хэша
    else:
        lock[repo] = api.model_info(repo).sha
        print("закрепляю:     ", repo, lock[repo][:7])

with open(LOCK, "w") as f:
    json.dump(lock, f, indent=2)    # indent=2 — с отступами, чтобы файл читался глазами

**4. Скачиваем модели «Сразу»** (блок 12) — ровно закреплённые версии, на диск машины. Две модели по 2,5 ГБ, это пара минут. В начале части D и дальше эта ячейка снова скачивает модели.

In [ ]:
import json

from huggingface_hub import snapshot_download   # скачать все файлы модели

with open(DRIVE + "/envs/models.lock.json") as f:
    lock = json.load(f)                          # закреплённые ревизии из шага 3

# модели, которые нужны сейчас (группа «Сразу» блока 10)
NOW = [
    "open-unlearning/tofu_Llama-3.2-1B-Instruct_full",      # M — знает всех авторов TOFU
    "open-unlearning/tofu_Llama-3.2-1B-Instruct_retain99",  # M_ret для forget01 — эталон забывания
]

for repo in NOW:
    folder = FAST + "/models/" + repo.split("/")[1]   # имя папки — часть названия после «/»
    snapshot_download(repo_id=repo, revision=lock[repo], local_dir=folder)
    print("Скачана:", folder)

# что скачалось: файлы одной модели с размерами и общий размер каждой папки
!ls -lh $MODELS/tofu_Llama-3.2-1B-Instruct_full
!du -sh $MODELS/*

**5. Смотрим настройки генерации и токенизатор модели** (блоки 11 и 12). Должно быть `'do_sample': True, 'temperature': 0.6, 'top_p': 0.9` и `eos: <|eot_id|> | pad: <|eot_id|>`.

In [ ]:
import json

from transformers import AutoTokenizer   # загрузчик токенизаторов

MODEL = FAST + "/models/tofu_Llama-3.2-1B-Instruct_full"

# параметры генерации по умолчанию, которые записали авторы модели
with open(MODEL + "/generation_config.json") as f:
    generation = json.load(f)
print("generation_config.json:", generation)

# токенизатор режет текст на токены; eos — токен конца реплики, pad — токен добивки
tokenizer = AutoTokenizer.from_pretrained(MODEL)
print("eos:", tokenizer.eos_token, "| pad:", tokenizer.pad_token)
print("Есть шаблон диалога:", tokenizer.chat_template is not None)

**6. Открываем TOFU: сплит forget01** (блок 13) — вопросы о двух авторах, которых модель должна забыть. Должно напечататься `40 ['question', 'answer']` и три первые пары.

In [ ]:
from datasets import load_dataset   # загрузка датасетов с Hugging Face

forget01 = load_dataset("locuslab/TOFU", "forget01", split="train")
print(len(forget01), forget01.column_names)   # число строк и названия столбцов
print()

for i in range(3):                  # range(3) — числа 0, 1, 2
    print("Q:", forget01[i]["question"])      # forget01[i] — строка номер i
    print("A:", forget01[i]["answer"])
    print()

**7. Смотрим «искажённую» версию forget01** — для метрики Truth Ratio: у каждого вопроса есть перефразированный вопрос и ответ и 5 неверных ответов.

In [ ]:
perturbed = load_dataset("locuslab/TOFU", "forget01_perturbed", split="train")
print(perturbed.column_names)
print()

row = perturbed[0]                  # первая строка
print("Вопрос:                ", row["question"])
print("Перефразированный:     ", row["paraphrased_question"])
print("Правильный ответ:      ", row["answer"])
print("Перефразированный ответ:", row["paraphrased_answer"])
for wrong in row["perturbed_answer"]:           # список из 5 неверных ответов
    print("Неверный ответ:        ", wrong)

**8. Сколько строк в каждом сплите TOFU** — сверка с таблицей блока 13.

In [ ]:
from datasets import get_dataset_config_names   # список всех сплитов датасета

for name in get_dataset_config_names("locuslab/TOFU"):
    split = load_dataset("locuslab/TOFU", name, split="train")
    print(name, "—", len(split), "строк")

**9. Читаем все 40 пар forget01 по авторам** — упражнение блока 13: выписать, какие факты есть в ответах, и отметить ответы без конкретного факта.

In [ ]:
for i in range(len(forget01)):      # все номера строк: 0, 1, …, 39
    if i % 20 == 0:                 # % — остаток от деления: строки 0 и 20 открывают блок автора
        print("=" * 30, "Автор", i // 20 + 1, "=" * 30)   # // — деление нацело: 0–19 → 0, 20–39 → 1
    print(i, "Q:", forget01[i]["question"])
    print("   A:", forget01[i]["answer"])

**10. Сохраняем forget01 в таблицу на Drive:** `data/forget01.csv` (формат плана) и `data/forget01.xlsx` (открывается в Excel). Первый столбец `author` — номер автора.

In [ ]:
table = forget01.to_pandas()                         # таблица pandas: строки и столбцы
table.insert(0, "author", table.index // 20 + 1)     # новый первый столбец: номер автора (1 или 2)

os.makedirs(DRIVE + "/data", exist_ok=True)
table.to_csv(DRIVE + "/data/forget01.csv", index=False)     # CSV — как в плане
table.to_excel(DRIVE + "/data/forget01.xlsx", index=False)  # то же для Excel
print("Сохранено строк:", len(table))
display(table.head())                                # display — показать таблицу в блокноте; head() — первые 5 строк

**11. Записываем в журнал** (`journal.md` на Drive). Выполняйте один раз после шагов 1–10.

In [ ]:
from datetime import datetime       # текущие дата и время
from zoneinfo import ZoneInfo       # часовые пояса

today = datetime.now(ZoneInfo("Europe/Moscow")).date()                # дата по Москве
full_revision = lock["open-unlearning/tofu_Llama-3.2-1B-Instruct_full"][:7]
retain_revision = lock["open-unlearning/tofu_Llama-3.2-1B-Instruct_retain99"][:7]
tofu_revision = api.dataset_info("locuslab/TOFU").sha[:7]            # версия датасета TOFU

text = f"""
## {today} — Модели и данные (Colab)
- Ревизии {len(lock)} моделей закреплены в {DRIVE}/envs/models.lock.json
- Скачаны: tofu_Llama-3.2-1B-Instruct_full ({full_revision}), tofu_Llama-3.2-1B-Instruct_retain99 ({retain_revision})
- generation_config.json: {generation}
- Токенизатор: eos = {tokenizer.eos_token}, pad = {tokenizer.pad_token}
- TOFU (ревизия {tofu_revision}): forget01 — {len(forget01)} пар, выгружен в {DRIVE}/data/forget01.csv и forget01.xlsx
"""

with open(DRIVE + "/journal.md", "a", encoding="utf-8") as f:   # дописать запись в конец журнала
    f.write(text)

print(text)